# REAM: Router-weighted Expert Activation Merging

[![GitHub](https://img.shields.io/badge/GitHub-JackBinary%2FREAM-blue)](https://github.com/JackBinary/REAM)

Compress Mixture-of-Experts LLMs by merging expert groups. Based on [Boris Knyazev's blog post](https://bknyaz.github.io/blog/2026/moe/), forked from [CerebrasResearch/reap](https://github.com/CerebrasResearch/reap).

**Requirements:** A100 GPU (40GB+) for Qwen3-30B-A3B. Smaller MoE models may work on T4/L4.

## 1. Install

In [ ]:
!git clone https://github.com/JackBinary/REAM.git
%cd REAM

In [ ]:
!pip install uv
!uv venv .venv --seed --python 3.12

import os
os.environ["VIRTUAL_ENV"] = os.path.join(os.getcwd(), ".venv")
os.environ["PATH"] = os.path.join(os.getcwd(), ".venv", "bin") + ":" + os.environ["PATH"]

In [ ]:
!git submodule init && git submodule update
!uv pip install --upgrade pip
!uv pip install setuptools wheel
!VLLM_USE_PRECOMPILED=1 uv pip install --editable . -vv --torch-backend auto

## 2. Configuration

Edit these parameters to match your setup.

In [ ]:
# ---- REAM Parameters ----
MODEL_NAME = "Qwen/Qwen3-30B-A3B"   # HuggingFace model ID
COMPRESSION_RATIO = 0.25              # Fraction of experts to remove (0.25 = 128 -> 96)
MAX_CLUSTER_SIZE = 16                 # Max experts per merge group
SEED = 42

# ---- Evaluation flags ----
RUN_LM_EVAL = True
RUN_EVALPLUS = True
RUN_LIVE_CODE_BENCH = False
RUN_MATH = False

## 3. (Optional) HuggingFace Login

Required for gated models. Skip if your model is public.

In [ ]:
# Uncomment to log in:
# from huggingface_hub import login
# login()

## 4. Run REAM

In [ ]:
import subprocess, shlex, os

short_name = MODEL_NAME.split("/")[-1]
server_log = f"ream_{short_name}_{SEED}.log"

cmd = [
    "python", "src/reap/ream.py",
    "--compression-ratio", str(COMPRESSION_RATIO),
    "--model-name", MODEL_NAME,
    "--dataset-name", "ream_mixed",
    "--merge-method", "frequency_weighted_average",
    "--permute", "activation_weight",
    "--cluster-method", "agglomerative",
    "--profile", "false",
    "--server_log_file_name", server_log,
    "--vllm-port", "8000",
    "--expert-sim", "router_logits",
    "--distance_measure", "cosine",
    "--frequency-penalty", "false",
    "--merged-model-dir-name", f"ream-{COMPRESSION_RATIO}",
    "--cluster-description", "ream",
    "--max-cluster-size", str(MAX_CLUSTER_SIZE),
    "--do-eval", str(RUN_LM_EVAL or RUN_EVALPLUS).lower(),
    "--run-lm-eval", str(RUN_LM_EVAL).lower(),
    "--run-evalplus", str(RUN_EVALPLUS).lower(),
    "--run-livecodebench", str(RUN_LIVE_CODE_BENCH).lower(),
    "--run-math", str(RUN_MATH).lower(),
    "--seed", str(SEED),
]

print("Running:", shlex.join(cmd))
print("="*60)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

rc = process.wait()
if rc != 0:
    raise RuntimeError(f"REAM exited with code {rc}")
print("\nDone!")

## 5. Inspect the Merged Model

After REAM finishes, the merged model is saved under `results/`. Check the output directory:

In [ ]:
import glob

result_dirs = sorted(glob.glob("results/**/ream-*", recursive=True))
for d in result_dirs:
    print(d)

## 6. (Optional) Upload to HuggingFace Hub

In [ ]:
# Uncomment and fill in to push your merged model:
#
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="results/<YOUR_MODEL_DIR>",
#     repo_id="<YOUR_USERNAME>/<YOUR_REPO_NAME>",
#     repo_type="model",
# )